# Phase 9.5 — WaveGenerator Fix Training

**Goal:** Retrain WaveGenerator from scratch with all six fixes from Phase 9 failure.

| Fix # | Root Cause | Solution |
|-------|-----------|----------|
| 1 | Context collapse (avg cosine 0.980) | L2-norm + LayerNorm + noise (σ=0.1) + 5× contrastive |
| 2 | Fixed 40 chunks from position 0 | Random start + variable length (5–40) |
| 3 | Batch size 1 → 17% GPU util | DataLoader batch_size=128, batched GRU |
| 4 | Scheduled sampling at 0% | Start at 50%, ramp to 90% |
| 5 | Train-inference mismatch | Gaussian jitter on precomputed contexts |
| 6 | Silent error swallowing | Traceback logging, abort on >10% skip rate |

**Architecture:** WaveGeneratorV3 — GRU batch_first=True, 3,713,570 params

**Stages:**
1. Load Phase 9 checkpoint (keep CSE, Field, GR, TL, CGN, Memory, WaveChunker, WaveToText — discard WaveGenerator)
2. Precompute frozen pipeline outputs with FIX #2 (random windows)
3. WaveGenerator training (15K steps, batched) with FIX #1,3,4,5
4. Joint WG+WTT fine-tuning (3K steps) with FIX #6
5. Evaluation & save

---

## Cell 1 — Clone & Install

In [ ]:
# Cell 1 — Clone repo & install dependencies
# ─────────────────────────────────────────────
# NOTE: faiss-cpu used instead of faiss-gpu to avoid install errors on Colab.
# faiss-gpu requires specific CUDA toolkit versions that often conflict.
# faiss-cpu works fine — we don't use FAISS for training.
# ─────────────────────────────────────────────
# NOTE: setup.py is a directory-scaffolding script (creates checkpoints/, etc.)
# NOT a setuptools package definition — so `pip install -e .` would fail.
# We run it directly with `python setup.py` instead.
# ─────────────────────────────────────────────
import os

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = "/content/FLUX"

if os.path.exists(WORK_DIR):
    print("  ℹ Repo already cloned — pulling latest...")
    !cd /content/FLUX && git pull --ff-only
else:
    print("  ℹ Cloning FLUX repo...")
    !git clone {REPO_URL} /content/FLUX

# Run setup.py to create project directories (checkpoints/, results/, etc.)
# This is NOT a setuptools setup.py — it's a scaffolding script.
!cd /content/FLUX && python setup.py

# Install deps — swap faiss-gpu for faiss-cpu to avoid CUDA conflicts
!cd /content/FLUX && sed 's/faiss-gpu/faiss-cpu/g' requirements.txt > /tmp/req_fixed.txt && \
    pip install -q -r /tmp/req_fixed.txt

# Ensure datasets and huggingface_hub are installed
!pip install -q datasets huggingface_hub

print("\n  ✓ Installation complete")

## Cell 2 — Hardware, Secrets & Logger

In [ ]:
# Cell 2 — Hardware detection, HF_TOKEN, PhaseLogger
import sys, os, time, random, math, traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

# ── Path setup ──
WORK_DIR = Path("/content/FLUX")
sys.path.insert(0, str(WORK_DIR))
for _phase in ['phase1', 'phase1_5', 'phase2', 'phase2_5', 'phase3', 'phase3_5',
               'phase4', 'phase5', 'phase6', 'phase7', 'phase8', 'phase8_5',
               'phase9', 'phase9_5']:
    _p = str(WORK_DIR / 'phases' / _phase)
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── Device ──
from flux_utils import get_device, PhaseLogger, PhaseResults, CHECKPOINT_DIR
device = get_device()
print(f"  Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── HF Token ──
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print(f"  ✓ HF_TOKEN loaded from Colab secrets")
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', None)
    if HF_TOKEN:
        print(f"  ✓ HF_TOKEN loaded from environment")
    else:
        print(f"  ⚠ No HF_TOKEN found — uploads will be skipped")

# ── Logger ──
log = PhaseLogger(phase=9.5)
log.separator("Phase 9.5: WaveGenerator Fix Training")
log.info(f"Device: {device}")

# ── Config ──
from train_wave_gen_v2 import PHASE9_5_CONFIG
print(f"\n  Phase 9.5 Config:")
for k, v in PHASE9_5_CONFIG.items():
    print(f"    {k}: {v}")

## Cell 3 — Smoke Test: Load Phase 9 Checkpoint

Load all frozen components from `phase9.phase.pt`.
Verify bridges are correctly patched, round-trip cosine is healthy.

In [ ]:
# Cell 3 — Smoke test: load Phase 9 checkpoint & verify frozen components
log.cell_start("Cell 3 — Smoke Test")

from train_wave_gen_v2 import load_from_phase9_checkpoint, build_fresh_wave_generator

model, chunker, wtt, ckpt = load_from_phase9_checkpoint(device=device)

# ── Verify bridges ──
with torch.no_grad():
    _fw = model.field.wave_to_feature.weight  # [512, 432]
    _bw = model.wave_to_field.weight           # [512, 432]
    _cos = F.cosine_similarity(_fw.flatten().unsqueeze(0), _bw.flatten().unsqueeze(0)).item()
    print(f"  wave_to_field == field.wave_to_feature: cosine {_cos:.6f}")
    assert _cos > 0.999, f"Bridge mismatch: cosine {_cos}"
    print(f"  ✓ Bridges correctly patched")

# ── Round-trip test ──
with torch.no_grad():
    _test = torch.randn(5, 432, device=device)
    _proj = model.wave_to_field(_test)
    _back = model.field_to_wave(_proj)
    _rt_cos = F.cosine_similarity(_test, _back, dim=-1).mean().item()
    print(f"  Round-trip cosine: {_rt_cos:.3f}")

# ── CSE encode test ──
with torch.no_grad():
    test_wave = model.cse.encode("The future of artificial intelligence")
    print(f"  CSE encode: full={list(test_wave.full.shape)}")
    assert test_wave.full.shape[1] == 432

# ── Field query test ──
n_attractors = model.field.num_attractors()
print(f"  Field: {n_attractors:,} attractors")

# ── Frozen param count check ──
frozen_params = sum(p.numel() for p in model.parameters())
frozen_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  FLUXModel: {frozen_params:,} params, {frozen_trainable} trainable (should be 0)")
assert frozen_trainable == 0, "Model should be fully frozen!"

print(f"\n  ✓ Smoke test passed — Phase 9 components healthy")
log.cell_end("Cell 3 — Smoke Test", "PASS")

## Cell 4 — Load Training Data

In [ ]:
# Cell 4 — Load training data
log.cell_start("Cell 4 — Load Data")

from train_wave_gen_v2 import load_training_data

texts = load_training_data(max_docs=10000)
split_idx = int(len(texts) * 0.9)
train_texts = texts[:split_idx]
eval_texts = texts[split_idx:]

print(f"  Total:  {len(texts):,} documents")
print(f"  Train:  {len(train_texts):,}")
print(f"  Eval:   {len(eval_texts):,}")
print(f"  Sample: {train_texts[0][:100]}...")

log.cell_end("Cell 4 — Load Data", "PASS")

---
## Stage 1: Build Fresh WaveGeneratorV3

WaveGenerator state from Phase 9 is DISCARDED. We build a randomly
initialized WaveGeneratorV3 with:
- `batch_first=True` GRU for [batch, seq, 864] processing
- `context_projection`: LayerNorm + Linear decorrelation
- Dropout 0.15 on GRU output
- 3,713,570 trainable parameters

### Cell 5a — WG Fresh (Build Model)

In [ ]:
# Cell 5a — Build fresh WaveGeneratorV3
log.cell_start("Cell 5a — WG Fresh")

generator = build_fresh_wave_generator(device=device, config=PHASE9_5_CONFIG)

# Architecture summary
n_params = sum(p.numel() for p in generator.parameters())
n_trainable = sum(p.numel() for p in generator.parameters() if p.requires_grad)
print(f"\n  WaveGeneratorV3 Architecture:")
print(f"    Total params:     {n_params:,}")
print(f"    Trainable:        {n_trainable:,}")
print(f"    GRU input:        864 (prev_wave:432 + context_wave:432)")
print(f"    GRU hidden:       {PHASE9_5_CONFIG['gru_hidden']}")
print(f"    GRU layers:       {PHASE9_5_CONFIG['gru_layers']}")
print(f"    Dropout:          {PHASE9_5_CONFIG['dropout']}")
print(f"    batch_first:      True")

# Verify forward_batch exists
assert hasattr(generator, 'forward_batch'), "Missing forward_batch method!"
assert hasattr(generator, 'process_context'), "Missing process_context method!"
assert hasattr(generator, 'init_hidden_batch'), "Missing init_hidden_batch method!"
print(f"  ✓ All V3 methods present")

log.cell_end("Cell 5a — WG Fresh", "PASS")

### Cell 5b — Precompute Pipeline Outputs

**FIX #2:** Random start position + variable sequence length (5–40 chunks).
Phase 9 used fixed 40 chunks from position 0 for ALL samples.

Pipeline: CSE → wave_to_field → GR → CGN → field.query → merged [512]

In [ ]:
# Cell 5b — Precompute frozen pipeline outputs (FIX #2)
log.cell_start("Cell 5b — Precompute")

from train_wave_gen_v2 import precompute_wg_data

precomputed = precompute_wg_data(
    model, chunker, train_texts,
    max_samples=8500, device=device,
)

print(f"\n  Precomputed: {len(precomputed):,} samples")
if len(precomputed) > 0:
    m, tw = precomputed[0]
    print(f"  Sample 0: merged={list(m.shape)}, target_waves={list(tw.shape)}")
    assert m.shape[0] == 512, f"Expected merged dim 512, got {m.shape[0]}"
    assert tw.shape[1] == 432, f"Expected wave dim 432, got {tw.shape[1]}"

log.cell_end("Cell 5b — Precompute", "PASS")

### Cell 5c — Precompute Diagnostic

Verify precomputed data quality before training.

In [ ]:
# Cell 5c — Precomputed Data Sanity Check
log.cell_start("Cell 5c — Precompute Diag")
import numpy as np

n = len(precomputed)
assert n > 0, "No precomputed samples!"

# 1. Shape check
m0, tw0 = precomputed[0]
print(f"  📐 Shape Check (sample 0):")
print(f"    merged:       {list(m0.shape)}  dtype={m0.dtype}")
print(f"    target_waves: {list(tw0.shape)}  dtype={tw0.dtype}")

# 2. Chunk length distribution (FIX #2 verification)
chunk_counts = [precomputed[i][1].shape[0] for i in range(n)]
print(f"\n  📊 Target Waves Per Sample (FIX #2: should be variable):")
print(f"    Min:    {min(chunk_counts)}")
print(f"    Max:    {max(chunk_counts)}")
print(f"    Mean:   {np.mean(chunk_counts):.1f}")
print(f"    Std:    {np.std(chunk_counts):.1f}")
if np.std(chunk_counts) < 1.0:
    print(f"    ⚠ All samples have same length — FIX #2 may not be working!")
else:
    print(f"    ✓ Variable lengths confirmed (FIX #2 active)")

# 3. Context diversity — KEY metric from Phase 9
sample_n = min(20, n)
merged_vecs = torch.stack([precomputed[i][0] for i in range(sample_n)])
cos_matrix = F.cosine_similarity(
    merged_vecs.unsqueeze(1), merged_vecs.unsqueeze(0), dim=-1
)
mask = ~torch.eye(sample_n, dtype=torch.bool)
off_diag = cos_matrix[mask]
avg_cos = off_diag.mean().item()
print(f"\n  📊 Merged Context Diversity (first {sample_n} samples):")
print(f"    Pairwise cosine: avg={avg_cos:.3f}  min={off_diag.min().item():.3f}  max={off_diag.max().item():.3f}")
print(f"    Phase 9 was: 0.980 (collapsed)")
if avg_cos > 0.95:
    print(f"    ⚠ Contexts still very similar — FIX #1 noise will help during training")
else:
    print(f"    ✓ Good diversity in raw contexts")

# 4. NaN/Inf check
nan_count = 0
inf_count = 0
for i in range(min(100, n)):
    for t in precomputed[i]:
        if torch.isnan(t).any(): nan_count += 1
        if torch.isinf(t).any(): inf_count += 1
print(f"\n  🔍 NaN/Inf Check: nan={nan_count}, inf={inf_count}")
assert nan_count == 0, f"Found {nan_count} NaN tensors!"
assert inf_count == 0, f"Found {inf_count} Inf tensors!"
print(f"    ✓ All clean")

log.cell_end("Cell 5c — Precompute Diag", "PASS")
print("\n  ✓ Precomputed data healthy — ready for training")

---
## Stage 2: WaveGenerator Training

Train WaveGeneratorV3 with batched GRU forward pass.

**Fixes active:**
- FIX #1: Noise augmentation on contexts (σ=0.1)
- FIX #3: batch_size=128 DataLoader (was 1)
- FIX #4: Scheduled sampling 50% → 90% (was 0% → ?)
- FIX #5: Context jitter for train-inference matching

Loss = MSE + cosine + 5× contrastive Wave 0 loss

### Cell 6a — WG Train (~15K steps)

In [ ]:
# Cell 6a — Stage 2: Train WaveGenerator (Batched)
log.cell_start("Cell 6a — WG Train")

from train_wave_gen_v2 import train_wave_generator

wg_result = train_wave_generator(
    generator, precomputed,
    max_steps=15000,
    batch_size=128,
    lr=3e-4,
    ss_start=0.5,
    ss_end=0.9,
    contrastive_weight=5.0,
    noise_sigma=0.1,
    log_interval=500,
    device=device,
    log=log,
)

print(f"\n  Training Summary:")
print(f"    Steps:          {wg_result.total_steps:,}")
print(f"    Final loss:     {wg_result.final_loss:.4f}")
print(f"    Avg loss:       {wg_result.avg_loss:.4f}")
print(f"    Cosine acc:     {wg_result.wave_cosine_accuracy:.3f}")
print(f"    Speed:          {wg_result.steps_per_second:.1f} step/s")
print(f"    Time:           {wg_result.total_time_seconds:.0f}s ({wg_result.total_time_seconds/60:.1f} min)")

log.metric("wg_final_loss", f"{wg_result.final_loss:.4f}")
log.metric("wg_cos_acc", f"{wg_result.wave_cosine_accuracy:.3f}")
log.metric("wg_speed", f"{wg_result.steps_per_second:.1f} step/s")
log.cell_end("Cell 6a — WG Train", "PASS")

### Cell 6b — WG Diagnostic

Test teacher-forced prediction, context dependency (Wave 0 should differ per context),
and free generation quality.

In [ ]:
# Cell 6b — WaveGenerator Diagnostic
log.cell_start("Cell 6b — WG Diag")

generator.eval()
wtt.eval()

# ── Teacher-forced test ──
print("  📊 Teacher-Forced Prediction (5 samples)")
print("  " + "=" * 60)

tf_cosines = []
for i in range(min(5, len(precomputed))):
    merged_cpu, target_cpu = precomputed[i]
    merged = merged_cpu.to(device)
    target = target_cpu.to(device)

    with torch.no_grad():
        predicted, confs = generator(merged, target)
        n_waves = min(len(predicted), len(target))
        cos_per = F.cosine_similarity(predicted[:n_waves], target[:n_waves], dim=-1)
        avg_cos = cos_per.mean().item()
        tf_cosines.append(avg_cos)
        print(f"  Sample {i}: {n_waves} waves, avg cosine={avg_cos:.3f}, "
              f"range=[{cos_per.min().item():.3f}, {cos_per.max().item():.3f}]")

print(f"\n  Teacher-forced avg cosine: {sum(tf_cosines)/len(tf_cosines):.3f}")

# ── Context dependency test (KEY Phase 9.5 metric) ──
print("\n  📊 Context Dependency Test (Wave 0)")
print("  " + "=" * 60)
if len(precomputed) >= 3:
    m0 = precomputed[0][0].to(device)
    m1 = precomputed[1][0].to(device)
    m2 = precomputed[2][0].to(device)
    t_dummy = precomputed[0][1][:3].to(device)

    with torch.no_grad():
        p0, _ = generator(m0, t_dummy)
        p1, _ = generator(m1, t_dummy)
        p2, _ = generator(m2, t_dummy)
        cos_01 = F.cosine_similarity(p0[0].unsqueeze(0), p1[0].unsqueeze(0)).item()
        cos_02 = F.cosine_similarity(p0[0].unsqueeze(0), p2[0].unsqueeze(0)).item()
        cos_12 = F.cosine_similarity(p1[0].unsqueeze(0), p2[0].unsqueeze(0)).item()
        avg_cross = (cos_01 + cos_02 + cos_12) / 3
        print(f"  Wave 0 cross-context cosines: {cos_01:.3f}, {cos_02:.3f}, {cos_12:.3f}")
        print(f"  Average: {avg_cross:.3f} (Phase 9 was 1.000, target <0.85)")
        if avg_cross < 0.85:
            print(f"  ✓ FIX #1 working — Wave 0 depends on context!")
        else:
            print(f"  ⚠ Context collapse may still be present")

    # Hidden init diversity
    with torch.no_grad():
        if hasattr(generator, 'init_hidden_batch'):
            h0 = generator.init_hidden_batch(m0.unsqueeze(0)).squeeze()
            h1 = generator.init_hidden_batch(m1.unsqueeze(0)).squeeze()
            h2 = generator.init_hidden_batch(m2.unsqueeze(0)).squeeze()
        else:
            h0 = generator.context_to_hidden(m0)
            h1 = generator.context_to_hidden(m1)
            h2 = generator.context_to_hidden(m2)

        h_cos_01 = F.cosine_similarity(h0.flatten().unsqueeze(0), h1.flatten().unsqueeze(0)).item()
        h_cos_02 = F.cosine_similarity(h0.flatten().unsqueeze(0), h2.flatten().unsqueeze(0)).item()
        h_cos_12 = F.cosine_similarity(h1.flatten().unsqueeze(0), h2.flatten().unsqueeze(0)).item()
        avg_h = (h_cos_01 + h_cos_02 + h_cos_12) / 3
        print(f"\n  Hidden init cross-context: {h_cos_01:.3f}, {h_cos_02:.3f}, {h_cos_12:.3f}")
        print(f"  Average: {avg_h:.3f} (Phase 9 was 0.996, target <0.90)")
        if avg_h < 0.90:
            print(f"  ✓ Hidden states are context-dependent")
        else:
            print(f"  ⚠ Hidden init may still be too similar")

# ── Free generation test ──
print("\n  📊 Free Generation (3 prompts)")
print("  " + "=" * 60)
from train_wave_gen_v2 import generate_text

gen_prompts = [
    "The future of artificial intelligence",
    "Scientists have discovered",
    "In the beginning",
]

for prompt in gen_prompts:
    try:
        result = generate_text(
            prompt, model, chunker, generator, wtt,
            max_waves=15, temperature=0.8,
        )
        continuation = result[len(prompt):].strip()
        print(f"  Prompt: {prompt}")
        print(f"  Output: {continuation[:120]}")
        print()
    except Exception as e:
        print(f"  ✗ {prompt[:30]}... → ERROR: {e}")
        print()

log.cell_end("Cell 6b — WG Diag", "PASS")

### Cell 6c — Context Diversity Evaluation

Formal evaluation of FIX #1 effectiveness.
Compare against Phase 9 baselines.

In [ ]:
# Cell 6c — Context Diversity Evaluation
log.cell_start("Cell 6c — Diversity Eval")

from train_wave_gen_v2 import evaluate_context_diversity

diversity = evaluate_context_diversity(generator, precomputed, device=device)

print(f"\n  ┌────────────────────────────────────────────────────┐")
print(f"  │  Phase 9.5 vs Phase 9 Comparison                   │")
print(f"  ├───────────────────────────┬──────────┬──────────────┤")
print(f"  │ Metric                    │ Phase 9  │ Phase 9.5    │")
print(f"  ├───────────────────────────┼──────────┼──────────────┤")
print(f"  │ Context avg cosine        │  0.980   │  {diversity['processed_ctx_avg_cosine']:.3f}        │")
print(f"  │ Wave 0 cross-ctx cosine   │  1.000   │  {diversity['wave0_cross_ctx_cosine']:.3f}        │")
print(f"  │ Hidden init cross-ctx     │  0.996   │  {diversity['hidden_init_cross_ctx_cosine']:.3f}        │")
print(f"  └───────────────────────────┴──────────┴──────────────┘")

# Acceptance criteria
w0_pass = diversity['wave0_cross_ctx_cosine'] < 0.85
h_pass = diversity['hidden_init_cross_ctx_cosine'] < 0.90
print(f"\n  Acceptance: Wave 0 < 0.85: {'✓ PASS' if w0_pass else '✗ FAIL'}")
print(f"  Acceptance: Hidden < 0.90: {'✓ PASS' if h_pass else '✗ FAIL'}")

log.metric("wave0_cross_ctx", f"{diversity['wave0_cross_ctx_cosine']:.3f}")
log.metric("hidden_cross_ctx", f"{diversity['hidden_init_cross_ctx_cosine']:.3f}")
log.cell_end("Cell 6c — Diversity Eval", "PASS")

---
## Stage 3: Joint Fine-Tuning (WG + WTT)

Fine-tune WaveGenerator + WaveToText together.
WG predictions pass through WTT — text loss backpropagates into WG.

**FIX #6:** Full tracebacks logged, abort if >10% skip rate.
Never prints success markers for zero-work stages.

### Cell 7a — Joint FT Fresh

In [ ]:
# Cell 7a — Prepare for joint fine-tuning
log.cell_start("Cell 7a — Joint FT Fresh")

_wg_params = sum(p.numel() for p in generator.parameters())
_wtt_params = sum(p.numel() for p in wtt.parameters())
print(f"  WG params:    {_wg_params:,}")
print(f"  WTT params:   {_wtt_params:,}")
print(f"  Joint total:  {_wg_params + _wtt_params:,}")
print(f"  WG lr:        1e-4")
print(f"  WTT lr:       5e-5")
print(f"  Max skip:     10%")

log.cell_end("Cell 7a — Joint FT Fresh", "PASS")

### Cell 7b — Joint FT Train (~3K steps)

In [ ]:
# Cell 7b — Stage 3: Joint fine-tuning
log.cell_start("Cell 7b — Joint FT Train")

from train_wave_gen_v2 import train_joint_finetune

joint_result = train_joint_finetune(
    generator, wtt, model, chunker, train_texts, precomputed,
    max_steps=3000,
    lr_wg=1e-4,
    lr_wtt=5e-5,
    log_interval=500,
    max_skip_rate=0.10,
    device=device,
    log=log,
)

print(f"\n  Joint FT Summary:")
print(f"    Steps:          {joint_result.total_steps:,}")
print(f"    Skipped:        {joint_result.skipped_steps} ({joint_result.skipped_steps/max(joint_result.total_steps,1):.0%})")
print(f"    Final loss:     {joint_result.final_loss:.4f}")
print(f"    Cosine acc:     {joint_result.wave_cosine_accuracy:.3f}")
print(f"    WTT word acc:   {joint_result.wtt_word_accuracy:.1%}")
print(f"    Time:           {joint_result.total_time_seconds:.0f}s")

log.metric("joint_cos_acc", f"{joint_result.wave_cosine_accuracy:.3f}")
log.metric("joint_wtt_acc", f"{joint_result.wtt_word_accuracy:.1%}")
log.metric("joint_skipped", f"{joint_result.skipped_steps}")
log.cell_end("Cell 7b — Joint FT Train", "PASS")

### Cell 7c — Joint FT Diagnostic

In [ ]:
# Cell 7c — Joint Fine-Tuning Diagnostic
log.cell_start("Cell 7c — Joint FT Diag")

generator.eval()
wtt.eval()

print("  📊 Post-Joint Generation (5 prompts)")
print("  " + "=" * 60)

eval_prompts = [
    "The relationship between energy and matter",
    "Modern technology relies on",
    "In the year 2025, scientists proved that",
    "The history of mathematics reveals",
    "Research shows that quantum",
]

valid_words = 0
total_words = 0

for prompt in eval_prompts:
    try:
        result = generate_text(
            prompt, model, chunker, generator, wtt,
            max_waves=20, temperature=0.8,
        )
        continuation = result[len(prompt):].strip()
        words = continuation.split()
        for w in words:
            clean = w.strip('.,;:!?\'\"-()[]{}').lower()
            if clean.isalpha() and len(clean) >= 2:
                total_words += 1
                if len(clean) <= 15:
                    valid_words += 1
        print(f"  Prompt: {prompt}")
        print(f"  Output: {continuation[:150]}")
        print()
    except Exception as e:
        print(f"  ✗ {prompt[:30]}... → ERROR: {e}")
        print()

valid_rate = valid_words / max(total_words, 1)
print(f"  Valid word rate: {valid_rate:.1%} ({valid_words}/{total_words})")
log.metric("valid_word_rate", f"{valid_rate:.1%}")
log.cell_end("Cell 7c — Joint FT Diag", "PASS")

---
## Cell 8 — Save Checkpoint

In [ ]:
# Cell 8 — Save Phase 9.5 checkpoint
log.cell_start("Cell 8 — Save Checkpoint")

from train_wave_gen_v2 import save_phase9_5_checkpoint

ckpt_path = save_phase9_5_checkpoint(
    model, chunker, generator, wtt,
    wg_result, joint_result,
    diversity_metrics=diversity,
    valid_word_rate=valid_rate,
)

print(f"  ✓ Saved: {ckpt_path}")
_size = ckpt_path.stat().st_size / 1e6
print(f"  Size: {_size:.1f} MB")

log.cell_end("Cell 8 — Save Checkpoint", "PASS")

---
## Tests
Run all three Phase 9.5 tests. Each is a standalone script.

In [ ]:
# Cell 9 — Test 1: Context collapse fix
log.cell_start("Cell 9 — Test 1")
!cd /content/FLUX && python phases/phase9_5/test_phase9_5_test1.py
log.cell_end("Cell 9 — Test 1")

In [ ]:
# Cell 10 — Test 2: Training mechanics
log.cell_start("Cell 10 — Test 2")
!cd /content/FLUX && python phases/phase9_5/test_phase9_5_test2.py
log.cell_end("Cell 10 — Test 2")

In [ ]:
# Cell 11 — Test 3: Full pipeline generation
log.cell_start("Cell 11 — Test 3")
!cd /content/FLUX && python phases/phase9_5/test_phase9_5_test3.py
log.cell_end("Cell 11 — Test 3")

---
## Demos
Run Phase 9.5 demo scripts.

In [ ]:
# Cell 12 — Demo 1: Free generation quality comparison
log.cell_start("Cell 12 — Demo 1")
!cd /content/FLUX && python phases/phase9_5/demo_phase9_5_demo1.py
log.cell_end("Cell 12 — Demo 1")

In [ ]:
# Cell 13 — Demo 2: Context diversity visualization
log.cell_start("Cell 13 — Demo 2")
!cd /content/FLUX && python phases/phase9_5/demo_phase9_5_demo2.py
log.cell_end("Cell 13 — Demo 2")

---
## Cell 14 — Interactive Exploration

In [ ]:
# Cell 14 — Interactive text generation
log.cell_start("Cell 14 — Interactive")

prompts = [
    "The nature of consciousness",
    "Water flows downhill because",
    "The most important discovery in physics was",
    "Artificial intelligence will",
    "In a distant galaxy",
    "The fundamental laws of thermodynamics state that",
    "Quantum entanglement suggests",
]

print("  🌊 FLUX Phase 9.5 — Wave-Level Generation (Fixed)")
print("  " + "=" * 60)

for p in prompts:
    try:
        result = generate_text(
            p, model, chunker, generator, wtt,
            max_waves=25, temperature=0.8,
        )
        print(f"\n  Prompt: {p}")
        print(f"  →  {result}")
    except Exception as e:
        print(f"\n  ✗ {p[:30]}... → {e}")

log.cell_end("Cell 14 — Interactive", "PASS")

## Cell 15 — View Results

In [ ]:
# Cell 15 — Generate and view results
log.cell_start("Cell 15 — Results")

results = PhaseResults(phase=9.5, component_name="WaveGenerator Fix")
results.add_metric("Architecture", "WaveGeneratorV3 (batch_first GRU, 3.7M params)")
results.add_metric("Fixes applied", "#1 context, #2 windows, #3 batch, #4 SS, #5 jitter, #6 errors")
results.add_metric("WG training steps", wg_result.total_steps)
results.add_metric("WG final loss", f"{wg_result.final_loss:.4f}")
results.add_metric("WG cosine accuracy", f"{wg_result.wave_cosine_accuracy:.3f}")
results.add_metric("WG speed", f"{wg_result.steps_per_second:.1f} step/s")
results.add_metric("Wave 0 cross-ctx cosine", f"{diversity['wave0_cross_ctx_cosine']:.3f} (was 1.000)")
results.add_metric("Hidden init cross-ctx", f"{diversity['hidden_init_cross_ctx_cosine']:.3f} (was 0.996)")
if joint_result:
    results.add_metric("Joint training steps", joint_result.total_steps)
    results.add_metric("Joint skipped", joint_result.skipped_steps)
    results.add_metric("Joint cosine accuracy", f"{joint_result.wave_cosine_accuracy:.3f}")
    results.add_metric("Joint WTT accuracy", f"{joint_result.wtt_word_accuracy:.1%}")
results.add_metric("Valid word rate", f"{valid_rate:.1%}")
results.add_metric("Total time", f"{wg_result.total_time_seconds + (joint_result.total_time_seconds if joint_result else 0):.0f}s")

# Add tests
results.add_test(
    "Wave 0 cross-context < 0.85",
    passed=diversity['wave0_cross_ctx_cosine'] < 0.85,
    score=f"{diversity['wave0_cross_ctx_cosine']:.3f}",
    threshold="< 0.85",
)
results.add_test(
    "Hidden init cross-context < 0.90",
    passed=diversity['hidden_init_cross_ctx_cosine'] < 0.90,
    score=f"{diversity['hidden_init_cross_ctx_cosine']:.3f}",
    threshold="< 0.90",
)
results.add_test(
    "Training speed > 200 step/s",
    passed=wg_result.steps_per_second > 200,
    score=f"{wg_result.steps_per_second:.1f}",
    threshold="> 200",
)
results.add_test(
    "Valid word rate > 30%",
    passed=valid_rate > 0.3,
    score=f"{valid_rate:.1%}",
    threshold="> 30%",
)

results.save(str(WORK_DIR / 'phases' / 'phase9_5' / 'RESULTS_PHASE_9_5.md'))

# Display
_rpath = WORK_DIR / 'phases' / 'phase9_5' / 'RESULTS_PHASE_9_5.md'
if _rpath.exists():
    print(_rpath.read_text())

log.cell_end("Cell 15 — Results", "PASS")

## Cell 16 — View Log

In [ ]:
# Cell 16 — View full training log
_logpath = Path("/content/FLUX/logs/phase9.5.log")
# Also check alternate name
if not _logpath.exists():
    _logpath = Path("/content/FLUX/logs/phase9_5.log")

if _logpath.exists():
    _lines = _logpath.read_text().splitlines()
    print(f"  Log: {len(_lines)} lines")
    # Print last 80 lines
    for line in _lines[-80:]:
        print(line)
else:
    print("  ⚠ No log file found, checking logs/ dir:")
    !ls -la /content/FLUX/logs/

## Cell 17 — Upload Checkpoint & Logs

In [ ]:
# Cell 17 — Upload to HuggingFace Hub
log.cell_start("Cell 17 — Upload")

if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    HF_REPO = "UnseenGAP/FLUX"

    # Upload checkpoint
    _ckpt = Path("/content/FLUX/checkpoints/phase9_5.phase.pt")
    if _ckpt.exists():
        try:
            api.upload_file(
                path_or_fileobj=str(_ckpt),
                path_in_repo="checkpoints/phase9_5.phase.pt",
                repo_id=HF_REPO,
                repo_type="model",
            )
            print(f"  ✓ Checkpoint uploaded to HuggingFace: {HF_REPO}")
        except Exception as e:
            print(f"  ⚠ Checkpoint upload failed: {e}")

    # Upload log
    for _logname in ['phase9.5.log', 'phase9_5.log']:
        _logp = Path(f"/content/FLUX/logs/{_logname}")
        if _logp.exists():
            try:
                api.upload_file(
                    path_or_fileobj=str(_logp),
                    path_in_repo=f"logs/{_logname}",
                    repo_id=HF_REPO,
                    repo_type="model",
                )
                print(f"  ✓ Log uploaded: {_logname}")
            except Exception as e:
                print(f"  ⚠ Log upload failed: {e}")
            break

    # Upload results
    _res = Path("/content/FLUX/phases/phase9_5/RESULTS_PHASE_9_5.md")
    if _res.exists():
        try:
            api.upload_file(
                path_or_fileobj=str(_res),
                path_in_repo="phases/phase9_5/RESULTS_PHASE_9_5.md",
                repo_id=HF_REPO,
                repo_type="model",
            )
            print(f"  ✓ Results uploaded")
        except Exception as e:
            print(f"  ⚠ Results upload failed: {e}")
else:
    print("  ⚠ No HF_TOKEN — skipping upload")

log.cell_end("Cell 17 — Upload", "PASS")

## Cell 18 — Git Commit & Push

In [ ]:
# Cell 18 — Git commit results/logs and push to GitHub
log.cell_start("Cell 18 — Git Push")

import subprocess

os.chdir("/content/FLUX")

# Add results and logs
files_to_add = [
    "phases/phase9_5/RESULTS_PHASE_9_5.md",
    "logs/",
]

for f in files_to_add:
    _fp = Path(f)
    if _fp.exists():
        !git add {f}
        print(f"  Added: {f}")

# Commit
!git diff --cached --stat
!git commit -m "Phase 9.5: WaveGenerator fix training results" --allow-empty

# Push (requires auth — may need token)
if HF_TOKEN:
    # Use HF token for GitHub push if ssh not configured
    try:
        !git push origin main 2>&1 || echo "  ⚠ Push failed — may need SSH key or PAT"
    except Exception as e:
        print(f"  ⚠ Git push failed: {e}")
else:
    print("  ℹ Skipping git push (no credentials)")

log.cell_end("Cell 18 — Git Push", "PASS")

## Cell 19 — Save Colab Artifacts

In [ ]:
# Cell 19 — Copy artifacts to /content/output for Colab download
import shutil
_out = Path("/content/output")
_out.mkdir(exist_ok=True)

# Checkpoint
_ckpt = Path("/content/FLUX/checkpoints/phase9_5.phase.pt")
if _ckpt.exists():
    shutil.copy2(_ckpt, _out / "phase9_5.phase.pt")
    _sz = (_out / 'phase9_5.phase.pt').stat().st_size / 1e6
    print(f"  ✓ Checkpoint → {_out / 'phase9_5.phase.pt'} ({_sz:.1f} MB)")

# Results
_res = Path("/content/FLUX/phases/phase9_5/RESULTS_PHASE_9_5.md")
if _res.exists():
    shutil.copy2(_res, _out / "RESULTS_PHASE_9_5.md")
    print(f"  ✓ Results → {_out / 'RESULTS_PHASE_9_5.md'}")

# Log(s)
for _logname in ['phase9.5.log', 'phase9_5.log']:
    _log = Path(f"/content/FLUX/logs/{_logname}")
    if _log.exists():
        shutil.copy2(_log, _out / _logname)
        print(f"  ✓ Log → {_out / _logname}")

print(f"\n  All artifacts saved to {_out}")

---
## Cell 20 — Quick Continuation: Reload from Checkpoint

If the runtime restarts, run Cell 1 (clone/install), Cell 2 (setup), then
this cell to reload all trained models from the saved checkpoint.
No retraining needed.

In [ ]:
# Cell 20 — Quick continuation: reload Phase 9.5 from checkpoint
# Run this INSTEAD of Cells 3-8 if you already have a trained checkpoint.
# Requires: Cell 1 (clone/install) + Cell 2 (setup) to have been run first.

from train_wave_gen_v2 import load_phase9_5_modules, generate_text

# Try local checkpoint first, fall back to HuggingFace
_local = Path("/content/FLUX/checkpoints/phase9_5.phase.pt")
_hf_path = None

if not _local.exists():
    print("  ℹ Local checkpoint not found — downloading from HuggingFace...")
    try:
        from huggingface_hub import hf_hub_download
        _hf_path = hf_hub_download(
            repo_id="UnseenGAP/FLUX",
            filename="checkpoints/phase9_5.phase.pt",
            local_dir="/content/FLUX",
            token=HF_TOKEN,
        )
        print(f"  ✓ Downloaded checkpoint from HuggingFace")
    except Exception as e:
        print(f"  ✗ Could not download checkpoint: {e}")
        print(f"  → Run Cells 3-8 to train from scratch")
        raise

# Also need phase9.phase.pt for frozen components
_p9 = Path("/content/FLUX/checkpoints/phase9.phase.pt")
if not _p9.exists():
    print("  ℹ Phase 9 checkpoint not found — downloading from HuggingFace...")
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download(
            repo_id="UnseenGAP/FLUX",
            filename="checkpoints/phase9.phase.pt",
            local_dir="/content/FLUX",
            token=HF_TOKEN,
        )
        print(f"  ✓ Downloaded Phase 9 checkpoint from HuggingFace")
    except Exception as e:
        print(f"  ✗ Could not download Phase 9 checkpoint: {e}")
        raise

# Load everything
model, chunker, generator, wtt = load_phase9_5_modules(device=device)

# Quick generation test
print("\n  🌊 Quick generation test:")
for p in ["The nature of consciousness", "Scientists discovered that"]:
    result = generate_text(p, model, chunker, generator, wtt, max_waves=15)
    print(f"  {p}")
    print(f"  → {result}")
    print()

print("  ✓ Phase 9.5 reloaded — ready for further experimentation")

---
## Summary

Phase 9.5 retrains the WaveGenerator from scratch with six critical fixes:

| Fix | Root Cause | Solution | Verified By |
|-----|-----------|----------|-------------|
| #1 | Context collapse (cosine 0.980) | LayerNorm + noise σ=0.1 + 5× contrastive | Wave 0 cross-ctx < 0.85 |
| #2 | Fixed 40 chunks from pos 0 | Random start + variable length | Chunk std > 1.0 |
| #3 | Batch size 1 (17% GPU) | DataLoader batch=128, batched GRU | Speed > 200 step/s |
| #4 | SS at 0% | Start at 50%, ramp to 90% | SS schedule logged |
| #5 | Train-inference mismatch | Gaussian jitter on contexts | Noise σ=0.1 applied |
| #6 | Silent error swallowing | Traceback + abort >10% skip | Skip rate in results |

### Components

| Component | Status |
|-----------|--------|
| CSE (Phase 1) | Frozen ✓ |
| Field (Phase 2, 75K+ attractors) | Frozen ✓ |
| GR, TL, CGN (Phase 3-5) | Frozen ✓ |
| Memory (Phase 6) | Frozen ✓ |
| wave_to_field / field_to_wave | Patched from Phase 2 weights ✓ |
| WaveChunker (Phase 9) | Frozen ✓ |
| WaveToText (Phase 9) | Frozen → Unfrozen for joint FT ✓ |
| **WaveGeneratorV3** | **RETRAINED from scratch** ✓ |

### Quick Continuation
To resume without retraining: Run Cell 1 → Cell 2 → Cell 20.